In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex AI Model Garden - Kimi-K3 (Deployment)

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/community/model_garden/model_garden_pytorch_kimi_k3_deployment.ipynb">
      <img alt="Workbench logo" src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" width="32px"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fcommunity%2Fmodel_garden%2Fmodel_garden_pytorch_kimi_k3_deployment.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_pytorch_kimi_k3_deployment.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
</tr></tbody></table>

<!-- @publisher_model_name moonshotai/kimi-k3@kimi-k3 -->
<!-- @model_name Kimi-K3 -->

## Overview

This notebook demonstrates serving [Kimi-K3](https://huggingface.co/moonshotai/Kimi-K3) with [SGLang](https://github.com/sgl-project/sglang) on `a4-highgpu-8g` machines with NVIDIA B200 GPUs on Vertex AI.

Kimi-K3 is a state-of-the-art large language model from Moonshot AI, featuring advanced reasoning and tool-calling capabilities. This notebook deploys Kimi-K3 using multi-host GPU serving with DSPARK speculative decoding ([Kimi-K3-DSpark](https://huggingface.co/RadixArk/Kimi-K3-DSpark)).

### Objective

- Deploy Kimi-K3 with SGLang on GPU using multi-host serving and [Spot VMs](https://cloud.google.com/compute/docs/instances/spot) (Optional). Multi-host GPU serving is a preview feature.

### File a bug

File a bug on [GitHub](https://github.com/GoogleCloudPlatform/vertex-ai-samples/issues/new) if you encounter any issue with the notebook.

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage pricing](https://cloud.google.com/storage/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Before you begin

In [ ]:
# @title Request for quota

# @markdown To deploy with a4-highgpu-8g (8 x B200) machines, check that you have sufficient quota: [CustomModelServingB200GPUsPerProjectPerRegion](https://console.cloud.google.com/iam-admin/quotas?metric=aiplatform.googleapis.com%2Fcustom_model_serving_nvidia_b200_gpus). Find the available region(s) [here](https://cloud.google.com/vertex-ai/docs/general/locations#region_considerations).

# @markdown If you don't have sufficient quota, request for quota following the instructions at ["Request a quota adjustment"](https://cloud.google.com/docs/quotas/view-manage#requesting_higher_quota).

# @markdown You can also use Compute Engine reservations with Vertex Prediction following the instructions [here](https://cloud.google.com/vertex-ai/docs/predictions/use-reservations). Note that the GCE quota for the shared reservation will be managed separately. Shared reservation is the only GCE consumption mode.

In [ ]:
# @title Setup Google Cloud project

# @markdown 1. [Make sure that billing is enabled for your project](https://cloud.google.com/billing/docs/how-to/modify-project).

# @markdown 2. **[Optional]** Set region. If not set, the region will be set automatically according to Colab Enterprise environment.

REGION = ""  # @param {type:"string"}

# Upgrade Vertex AI SDK.
! pip3 install --upgrade --quiet 'google-cloud-aiplatform==1.103.0'

# Import the necessary packages
import importlib
import os
import time
from typing import Tuple

import requests
from google import auth
from google.cloud import aiplatform

# Upgrade Vertex AI SDK.
if os.environ.get("VERTEX_PRODUCT") != "COLAB_ENTERPRISE":
    ! pip install --upgrade tensorflow
! git clone https://github.com/GoogleCloudPlatform/vertex-ai-samples.git

common_util = importlib.import_module(
    "vertex-ai-samples.notebooks.community.model_garden.docker_source_codes.notebook_util.common_util"
)

LABEL = "sglang_gpu"
models, endpoints = {}, {}

# Get the default cloud project id.
PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]

# Get the default region for launching jobs.
if not REGION:
    REGION = os.environ["GOOGLE_CLOUD_REGION"]

# Initialize Vertex AI API.
print("Initializing Vertex AI API.")
aiplatform.init(project=PROJECT_ID, location=REGION)

! gcloud config set project $PROJECT_ID

import vertexai

vertexai.init(
    project=PROJECT_ID,
    location=REGION,
)

## Specify model artifacts

For a more reliable deployment, it is recommended to upload the Kimi-K3 base model (`moonshotai/Kimi-K3`) and its speculative draft model (`RadixArk/Kimi-K3-DSpark`) to a personal Google Cloud Storage (GCS) bucket beforehand.

In the cell below, you can optionally specify custom GCS paths to your uploaded model artifacts. If provided, the deployment will use your GCS paths; otherwise, it will default to the pre-staged model paths.

In [ ]:
# @title Specify custom GCS model paths

# @markdown **[Optional]** Specify custom GCS paths to your uploaded Kimi-K3 base model and speculative draft model artifacts. If left empty, default pre-staged paths will be used.
BASE_MODEL_GCS_URI = ""  # @param {type:"string"}
SPECULATIVE_DRAFT_MODEL_GCS_URI = ""  # @param {type:"string"}

BASE_MODEL_ID = "moonshotai/Kimi-K3"

if BASE_MODEL_GCS_URI and BASE_MODEL_GCS_URI.strip():
    base_model_path = BASE_MODEL_GCS_URI.strip()
    print(f"Using custom base model GCS path: {base_model_path}")
else:
    base_model_path = "gs://vertex-model-garden-restricted-us/moonshotai/Kimi-K3"
    print(f"Using default base model path: {base_model_path}")

if SPECULATIVE_DRAFT_MODEL_GCS_URI and SPECULATIVE_DRAFT_MODEL_GCS_URI.strip():
    speculative_draft_model_path = SPECULATIVE_DRAFT_MODEL_GCS_URI.strip()
    print(
        "Using custom speculative draft model GCS path:"
        f" {speculative_draft_model_path}"
    )
else:
    speculative_draft_model_path = "RadixArk/Kimi-K3-DSpark"
    print(
        "Using default speculative draft model path:" f" {speculative_draft_model_path}"
    )

## Deploy Kimi-K3 with SGLang

In [ ]:
# @title Deploy Kimi-K3 model on Vertex AI

# @markdown This section deploys the Kimi-K3 model to a Vertex AI Prediction Endpoint. It takes ~30 minutes to finish.

# @markdown The pre-built serving docker image for SGLang.
# @markdown The current B200 serving support in Model Garden is preliminary and will be continuously improved in the future.
SGLANG_DOCKER_URI = (
    "us-docker.pkg.dev/agent-platform-mg-public/containers/sglang-airlock:kimi-k3"
)

# @markdown Choose whether to use a [Spot VM](https://cloud.google.com/compute/docs/instances/spot) for the deployment.
is_spot = False  # @param {type:"boolean"}

# @markdown Set use_dedicated_endpoint to False if you don't want to use [dedicated endpoint](https://cloud.google.com/vertex-ai/docs/general/deployment#create-dedicated-endpoint). Note that [dedicated endpoint does not support VPC Service Controls](https://cloud.google.com/vertex-ai/docs/predictions/choose-endpoint-type), uncheck the box if you are using VPC-SC.
use_dedicated_endpoint = True  # @param {type:"boolean"}

# @markdown Find Vertex AI prediction supported accelerators and regions at https://cloud.google.com/vertex-ai/docs/predictions/configure-compute.
accelerator_type = "NVIDIA_B200"  # @param ["NVIDIA_B200"] {isTemplate:true}
if accelerator_type == "NVIDIA_B200":
    accelerator_count = 8
    machine_type = "a4-highgpu-8g"
else:
    raise ValueError("Sample deployment options are not available.")
multihost_gpu_node_count = 2

common_util.check_quota(
    project_id=PROJECT_ID,
    region=REGION,
    accelerator_type=accelerator_type,
    accelerator_count=int(accelerator_count * multihost_gpu_node_count),
    is_for_training=False,
    is_spot=is_spot,
)


def poll_operation(op_name: str) -> bool:
    creds, _ = auth.default()
    auth_req = auth.transport.requests.Request()
    creds.refresh(auth_req)
    headers = {
        "Authorization": f"Bearer {creds.token}",
    }
    get_resp = requests.get(
        f"https://{REGION}-aiplatform.googleapis.com/ui/{op_name}",
        headers=headers,
    )
    opjs = get_resp.json()
    if "error" in opjs:
        raise ValueError(f"Operation failed: {opjs['error']}")
    return opjs.get("done", False)


def poll_and_wait(op_name: str, total_wait: int, interval: int = 60):
    waited = 0
    while not poll_operation(op_name):
        if waited > total_wait:
            raise TimeoutError("Operation timed out")
        print(
            f"\rStill waiting for operation... Waited time in second: {waited:<6}",
            end="",
            flush=True,
        )
        waited += interval
        time.sleep(interval)


def deploy_model_kimi_k3_sglang(
    model_name: str,
    base_model_path: str,
    speculative_draft_model_path: str,
    machine_type: str,
    accelerator_type: str,
    accelerator_count: int,
    multihost_gpu_node_count: int,
    use_dedicated_endpoint: bool = False,
    is_spot: bool = False,
) -> Tuple[aiplatform.Model, aiplatform.Endpoint]:
    """Deploys Kimi-K3 with SGLang into Vertex AI."""
    endpoint = aiplatform.Endpoint.create(
        display_name=f"{model_name}-endpoint",
        dedicated_endpoint_enabled=use_dedicated_endpoint,
    )

    sglang_args = [
        "./entrypoint.sh",
        f"--model={base_model_path}",
        "--trust-remote-code",
        f"--tp-size={int(accelerator_count * multihost_gpu_node_count)}",
        "--mem-fraction-static=0.85",
        "--disable-flashinfer-autotune",
        "--enable-metrics",
        "--watchdog-timeout=3600",
        "--reasoning-parser=kimi_k3",
        "--tool-call-parser=kimi_k3",
        '--model-loader-extra-config={"enable_multithread_load": true}',
        "--mamba-full-memory-ratio=0.43",
        "--speculative-algorithm=DSPARK",
        f"--speculative-draft-model-path={speculative_draft_model_path}",
        "--speculative-dspark-block-size=7",
        "--enable-linear-replayssm-spec",
        "--enable-hierarchical-cache",
    ]

    env_vars = {
        "MODEL_ID": BASE_MODEL_ID,
        "DEPLOY_SOURCE": "notebook",
        "NCCL_DEBUG": "TRACE",
    }

    try:
        if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
            env_vars["HF_TOKEN"] = os.environ["HF_TOKEN"]
    except Exception:
        pass

    model = aiplatform.Model.upload(
        display_name=model_name,
        serving_container_image_uri=SGLANG_DOCKER_URI,
        serving_container_command=["./gcs_download_launcher.sh"],
        serving_container_args=sglang_args,
        serving_container_ports=[30000],
        serving_container_predict_route="/vertex_generate",
        serving_container_health_route="/health",
        serving_container_environment_variables=env_vars,
        serving_container_shared_memory_size_mb=(32 * 1024),  # 32768 MB
        serving_container_deployment_timeout=7200,
        model_garden_source_model_name="publishers/moonshotai/models/kimi-k3@kimi-k3",
    )
    print(
        f"Deploying {model_name} on {machine_type} with"
        f" {int(accelerator_count * multihost_gpu_node_count)} {accelerator_type}"
        " GPU(s)."
    )

    creds, _ = auth.default()
    auth_req = auth.transport.requests.Request()
    creds.refresh(auth_req)

    url = f"https://{REGION}-aiplatform.googleapis.com/ui/projects/{PROJECT_ID}/locations/{REGION}/endpoints/{endpoint.name}:deployModel"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {creds.token}",
    }
    data = {
        "deployedModel": {
            "model": model.resource_name,
            "displayName": model_name,
            "dedicatedResources": {
                "machineSpec": {
                    "machineType": machine_type,
                    "multihostGpuNodeCount": multihost_gpu_node_count,
                    "acceleratorType": accelerator_type,
                    "acceleratorCount": accelerator_count,
                },
                "minReplicaCount": 1,
                "maxReplicaCount": 1,
            },
            "system_labels": {
                "NOTEBOOK_NAME": "model_garden_pytorch_kimi_k3_deployment.ipynb",
                "NOTEBOOK_ENVIRONMENT": common_util.get_deploy_source(),
            },
        },
    }
    if is_spot:
        data["deployedModel"]["dedicatedResources"]["spot"] = True
    response = requests.post(url, headers=headers, json=data)
    print(f"Deploy Model response: {response.json()}")
    if response.status_code != 200 or "name" not in response.json():
        raise ValueError(f"Failed to deploy model: {response.text}")
    poll_and_wait(response.json()["name"], 7200)
    print("endpoint_name:", endpoint.name)

    return model, endpoint


models["sglang_gpu"], endpoints["sglang_gpu"] = deploy_model_kimi_k3_sglang(
    model_name=common_util.get_job_name_with_datetime(prefix="kimi-k3-serve"),
    base_model_path=base_model_path,
    speculative_draft_model_path=speculative_draft_model_path,
    machine_type=machine_type,
    accelerator_type=accelerator_type,
    accelerator_count=accelerator_count,
    multihost_gpu_node_count=multihost_gpu_node_count,
    use_dedicated_endpoint=use_dedicated_endpoint,
    is_spot=is_spot,
)

In [ ]:
# @title Raw predict


# @markdown Once deployment succeeds, you can send requests to the endpoint with text prompts. Sampling parameters supported by SGLang can be found [here](https://docs.sglang.ai/backend/sampling_params.html).

# @markdown Example:

# @markdown ```
# @markdown User: What is the best way to diagnose and fix a flickering light in my house?
# @markdown Assistant: Okay, so I need to figure out how to diagnose and fix a flickering light in my house. Hmm, where do I start? Let's think. First, I remember that flickering lights can be caused by various issues. Maybe the bulb is loose? That's a common problem. Let me start with the simplest things first.
# @markdown ```
# @markdown Additionally, you can moderate the generated text with Vertex AI. See [Moderate text documentation](https://cloud.google.com/natural-language/docs/moderating-text) for more details.

# Loads an existing endpoint instance using the endpoint name:
# - Using `endpoint_name = endpoint.name` allows us to get the
#   endpoint name of the endpoint `endpoint` created in the cell
#   above.
# - Alternatively, you can set `endpoint_name = "1234567890123456789"` to load
#   an existing endpoint with the ID 1234567890123456789.
# You may uncomment the code below to load an existing endpoint.

# endpoint_name = ""  # @param {type:"string"}
# aip_endpoint_name = (
#     f"projects/{PROJECT_ID}/locations/{REGION}/endpoints/{endpoint_name}"
# )
# endpoint = aiplatform.Endpoint(aip_endpoint_name)

prompt = "What is a car?"  # @param {type: "string"}
# @markdown If you encounter an issue like `ServiceUnavailable: 503 Took too long to respond when processing`, you can reduce the maximum number of output tokens, by lowering `max_tokens`.
max_new_tokens = 1024  # @param {type:"integer"}
temperature = 0.6  # @param {type:"number"}
top_p = 0.95  # @param {type:"number"}

# Overrides parameters for inferences.
instances = [{"text": prompt}]
parameters = {
    "sampling_params": {
        "max_new_tokens": max_new_tokens,
        "temperature": temperature,
        "top_p": top_p,
    }
}

response = endpoints["sglang_gpu"].predict(
    instances=instances, use_dedicated_endpoint=use_dedicated_endpoint
)

for prediction in response.predictions:
    print(prediction)

# @markdown Click "Show Code" to see more details.

## Clean up resources

In [ ]:
# @title Delete the models and endpoints

# @markdown  Delete the experiment models and endpoints to recycle the resources
# @markdown  and avoid unnecessary continuous charges that may incur.

# Undeploy model and delete endpoint.
for endpoint in endpoints.values():
    endpoint.delete(force=True)

# Delete models.
for model in models.values():
    model.delete()

delete_bucket = False  # @param {type:"boolean"}
if delete_bucket:
    ! gsutil -m rm -r $BUCKET_NAME